# Part 6 — PyCaret MLOps

This notebook follows the practical lifecycle of a PyCaret classification model:

**train → evaluate honestly → finalize → save/load → predict → monitor drift → retrain**

The important idea is that model training is only one part of machine learning.  
MLOps keeps the model reproducible, deployable, monitored, and refreshable.

## Colab runtime requirement

This notebook uses **PyCaret 3.3.2**.

Before running in Colab choose:

**Runtime → Change runtime type → Runtime Version → 2025.07**

This provides **Python 3.11.13**, which is compatible with PyCaret 3.3.2.


## 1. Install PyCaret 3.3.2

### What we saw before
- This notebook uses the PyCaret 3.x MLOps workflow.
- The software version must match the Python runtime.

### How this cell is different
- This cell only prepares the environment.
- It installs the core PyCaret package, not the larger optional extras.

### What this cell does
- Checks the active Python version.
- Gives a clear instruction if the wrong Colab runtime is active.
- Installs PyCaret 3.3.2.

### What we achieve after running it
- The environment is ready for training, persistence, monitoring, and retraining.

### Remember
Use Colab runtime **2025.07**.

**PyCaret 3.3.2 + Python 3.11**


In [ ]:
import sys
import subprocess

print("Python version:", sys.version.split()[0])

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "This notebook uses PyCaret 3.3.2. "
        "In Colab choose Runtime -> Change runtime type -> "
        "Runtime Version -> 2025.07, reconnect, then run again."
    )

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pycaret==3.3.2"
])

print("PyCaret 3.3.2 installation completed.")
print("If Colab asks for a runtime restart, restart once and continue.")


## 2. Create the Data and Train/Test Split

### What we saw before
- PyCaret is installed.
- We still need a dataset and a clearly defined target before training a model.

### How this cell is different
- This cell creates the data that will be used through the full MLOps lifecycle.
- The target is `overspent_next`, which predicts whether a household will overspend next month.

### What this cell does
- Creates household features such as income, rent, dining, groceries, and previous overspending behavior.
- Creates the binary target `overspent_next`.
- Splits the data into training and test sets.
- Defines a helper called `report_card()` for later monitoring comparisons.

### What we achieve after running it
- `train_df` is available for model development.
- `test_df` is kept separate for honest evaluation.
- We have a reusable metric function for monitoring.

### Remember
**Train data = model learns from it.**

**Test data = honest final check.**

The test set should remain unseen until evaluation.


In [ ]:
import os
import pathlib
import tempfile
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss
)

from pycaret.classification import *
from pycaret.classification import ClassificationExperiment

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
LABEL = "overspent_next"
rng = np.random.default_rng(RANDOM_STATE)

def make_households(n=600, seed=42):
    r = np.random.default_rng(seed)

    city_tier = r.choice(
        ["metro", "city", "town"],
        size=n,
        p=[0.35, 0.40, 0.25]
    )

    household_size = r.integers(1, 6, n)
    income = r.lognormal(mean=8.45, sigma=0.35, size=n)

    rent_ratio = pd.Series(city_tier).map({
        "metro": 0.34,
        "city": 0.27,
        "town": 0.20
    }).to_numpy()

    rent = income * rent_ratio * r.normal(1.0, 0.05, n)
    groceries = (180 + 90 * household_size) * r.normal(1.0, 0.12, n)
    dining = r.gamma(2.0, 70, n)
    utilities = (60 + 25 * household_size) * r.normal(1.0, 0.10, n)

    total_spend = rent + groceries + dining + utilities
    avg_spend_prev = total_spend * r.normal(0.97, 0.08, n)

    overspend_rate_prev = np.clip(
        (total_spend / income - 0.45) + r.normal(0, 0.08, n),
        0,
        1
    )

    logit = (
        -3.0
        + 4.3 * overspend_rate_prev
        + 2.0 * (total_spend / income)
        + 0.35 * (city_tier == "metro")
        + 0.15 * household_size
    )

    p = 1 / (1 + np.exp(-logit))
    overspent_next = (r.random(n) < p).astype(int)

    return pd.DataFrame({
        "city_tier": city_tier,
        "household_size": household_size,
        "income": income.round(0),
        "rent": rent.round(0),
        "groceries": groceries.round(0),
        "dining": dining.round(0),
        "utilities": utilities.round(0),
        "avg_spend_prev": avg_spend_prev.round(0),
        "overspend_rate_prev": overspend_rate_prev.round(3),
        LABEL: overspent_next
    })

df = make_households()

train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df[LABEL]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

def report_card(y_true, y_pred, y_prob, name):
    return pd.DataFrame({
        "accuracy": [accuracy_score(y_true, y_pred)],
        "precision": [precision_score(y_true, y_pred, zero_division=0)],
        "recall": [recall_score(y_true, y_pred, zero_division=0)],
        "f1": [f1_score(y_true, y_pred, zero_division=0)],
        "roc_auc": [roc_auc_score(y_true, y_prob)],
        "log_loss": [log_loss(y_true, y_prob)]
    }, index=[name])

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Positive rate:", round(df[LABEL].mean(), 3))

display(train_df.head())


## 3. Train the Model and Freeze Honest Metrics

### What we saw before
- We created separate training and test datasets.
- The held-out test set has not been used for training.

### How this cell is different
- Now we build the actual PyCaret classification experiment.
- We also save the model's honest test performance **before finalization**.

### What this cell does
- `setup()` creates the PyCaret experiment.
- `test_data=test_df` makes the test set explicit.
- `compare_models()` compares Logistic Regression, Random Forest, and Gradient Boosting.
- Models are ranked using AUC.
- `predict_model(best)` evaluates the selected model on the held-out test set.
- `pull()` saves those metrics into `before`.

### What we achieve after running it
- We get the best model from cross-validation.
- We also freeze the honest held-out metrics before deployment preparation begins.

### Remember
This is one of the most important MLOps rules:

**Evaluate first → save the honest metrics → finalize later.**

Once the model is finalized on all labelled data, the original test set is no longer unseen.


In [ ]:
exp = setup(
    data=train_df,
    target=LABEL,
    test_data=test_df,
    index=False,
    session_id=RANDOM_STATE,
    fold=2,
    verbose=False,
    html=False
)

best = compare_models(
    include=["lr", "rf"],
    fold=2,
    sort="AUC",
    verbose=False
)

comparison = pull()

print("Model comparison:")
display(
    comparison[
        ["Model", "Accuracy", "AUC", "Recall", "Prec.", "F1", "TT (Sec)"]
    ]
)

_ = predict_model(best, verbose=False)
before = pull().iloc[0].copy()

print("\nHonest held-out metrics BEFORE finalization:")
display(before.to_frame("value"))

best_id = comparison.index[0]
best_name = comparison.loc[best_id, "Model"]

print("Best model:", best_name)
print("PyCaret model id:", best_id)


## 4. Finalize the Model

### What we saw before
- The best model was selected using cross-validation.
- Its held-out test performance was saved in `before`.

### How this cell is different
- `finalize_model()` prepares the model for deployment.
- It refits the selected pipeline using **all available labelled data**, including the previous holdout rows.

### What this cell does
- Calls `finalize_model(best)`.
- Creates a complete pipeline containing preprocessing and the estimator.
- Demonstrates that scores calculated on the old test rows after finalization are no longer honest test scores.

### What we achieve after running it
- `final` is the model we would save and deploy.
- The original performance report remains stored separately in `before`.

### Remember
**Do not report test metrics calculated after `finalize_model()`.**

The finalized model has already seen those rows.

The clean lifecycle is:

**select model → evaluate once → freeze report → finalize model**


In [ ]:
final = finalize_model(best)

_ = predict_model(final, verbose=False)
after = pull().iloc[0].copy()

gap = pd.DataFrame({
    "Before finalize - honest holdout": before,
    "After finalize - no longer holdout": after
})

print("Comparison:")
display(gap)

print("\nFinal model type:", type(final).__name__)


## 5. Save and Load the Complete Pipeline

### What we saw before
- `finalize_model()` produced the deployment-ready pipeline.
- That pipeline includes both preprocessing and the trained estimator.

### How this cell is different
- Instead of keeping the model only in notebook memory, we persist it to disk.

### What this cell does
- `save_model()` saves the full PyCaret pipeline as a `.pkl` file.
- `load_model()` loads it back.
- The same raw rows are scored with the in-memory model and the loaded model.
- Their probabilities are compared.

### What we achieve after running it
- We confirm that the saved model can be restored without rewriting preprocessing logic.
- Identical predictions show that the persisted pipeline represents the same model.

### Remember
`save_model()` saves more than the estimator.

It saves the **complete prediction pipeline**:

**raw data → preprocessing → model → prediction**

Also record your PyCaret and scikit-learn versions with the model artifact.


In [ ]:
MODEL_DIR = pathlib.Path(
    tempfile.mkdtemp(prefix="pycaret_mlops_")
)

model_path = MODEL_DIR / "overspend_pipeline"

saved_pipeline, saved_file = save_model(
    final,
    str(model_path),
    verbose=False
)

loaded = load_model(
    str(model_path),
    verbose=False
)

X_test = test_df.drop(columns=[LABEL])

p_memory = predict_model(
    final,
    data=X_test,
    raw_score=True,
    verbose=False
)["prediction_score_1"]

p_disk = predict_model(
    loaded,
    data=X_test,
    raw_score=True,
    verbose=False
)["prediction_score_1"]

difference = np.abs(
    p_memory.to_numpy() - p_disk.to_numpy()
).max()

print("Saved file:", saved_file)
print("Loaded model type:", type(loaded).__name__)
print("Maximum prediction difference:", f"{difference:.2e}")


## 6. Predict on Completely New Data

### What we saw before
- The trained pipeline was finalized, saved, and loaded successfully.
- The loaded model reproduced the same predictions as the in-memory model.

### How this cell is different
- Now we use the saved model the way a real application would use it: on rows that were never part of training or testing.

### What this cell does
- Generates five new households using a different random seed.
- Removes the target column before scoring.
- Uses `predict_model(..., raw_score=True)` to return both class probabilities.
- Displays the predicted class and `prediction_score_1`.

### What we achieve after running it
- We demonstrate the actual inference step of an ML system.
- The persisted model can accept raw new rows and return predictions.

### Remember
With `raw_score=True`:

**`prediction_score_1` = probability of the positive class**

The ordinary `prediction_score` is the confidence of whichever class was predicted, so it should not automatically be interpreted as positive-class risk.


In [ ]:
new_households = make_households(
    n=5,
    seed=99
)

new_rows = new_households.drop(
    columns=[LABEL]
)

scored = predict_model(
    loaded,
    data=new_rows,
    raw_score=True,
    verbose=False
)

view = scored[
    [
        "city_tier",
        "income",
        "dining",
        "overspend_rate_prev",
        "prediction_label",
        "prediction_score_1"
    ]
].copy()

view["actual_outcome_for_demo"] = new_households[LABEL].values

display(view.round(3))


## 7. Monitor Data Drift with PSI

### What we saw before
- The saved model works on new observations.
- Deployment does not mean the work is finished.

### How this cell is different
- Instead of asking whether the model can predict, we ask whether the **incoming data still looks like the data the model was developed on**.

### What this cell does
- Uses the held-out data as a reference month.
- Creates a drifted month by increasing income and dining values.
- Calculates the Population Stability Index, or PSI, for every numeric feature.
- Classifies each feature as:
  - `stable`
  - `watch`
  - `act`

### What we achieve after running it
- We identify which incoming features have changed significantly.
- This gives an early warning even before the new month's true outcomes are available.

### Remember
Three different things can drift:

**Data drift** → input distribution changes

**Label drift** → positive/negative rate changes

**Concept drift** → relationship between inputs and target changes

PSI mainly detects **data distribution drift**.

A common rule of thumb is:

**PSI < 0.10 → stable**

**0.10 to 0.25 → watch**

**> 0.25 → investigate/action**


In [ ]:
reference = test_df.copy()
drifted = reference.copy()

# Simulate a changed month
drifted["income"] = (
    drifted["income"] * 1.20
).round(0)

drifted["dining"] = (
    drifted["dining"] * 1.35 + 40
).round(0)

def psi(ref, cur, bins=10):
    ref = pd.Series(ref).dropna()
    cur = pd.Series(cur).dropna()

    edges = np.unique(
        np.quantile(
            ref,
            np.linspace(0, 1, bins + 1)
        )
    )

    edges[0] = -np.inf
    edges[-1] = np.inf

    r = np.histogram(ref, edges)[0] / len(ref)
    c = np.histogram(cur, edges)[0] / len(cur)

    r = np.clip(r, 1e-4, None)
    c = np.clip(c, 1e-4, None)

    return float(
        np.sum(
            (c - r) * np.log(c / r)
        )
    )

numeric_features = [
    c for c in reference.columns
    if c != LABEL
    and pd.api.types.is_numeric_dtype(reference[c])
    and reference[c].nunique() > 2
]

psi_table = pd.DataFrame({
    "PSI": {
        col: psi(
            reference[col],
            drifted[col]
        )
        for col in numeric_features
    }
}).sort_values(
    "PSI",
    ascending=False
)

psi_table["status"] = np.select(
    [
        psi_table["PSI"] > 0.25,
        psi_table["PSI"] > 0.10
    ],
    [
        "act",
        "watch"
    ],
    default="stable"
)

display(psi_table.round(4))


## 8. Monitor Model Scores and Performance

### What we saw before
- PSI showed which input features changed.
- But a feature can drift without strongly affecting the model.

### How this cell is different
- Now we monitor the model's **predictions themselves**.
- This tells us whether the data shift is actually changing model behavior.

### What this cell does
- Scores both the reference month and drifted month using the original non-finalized `best` model.
- Compares the mean predicted probability and the percentage of rows flagged at threshold 0.5.
- Once labels are assumed to be available, calculates Accuracy, Precision, Recall, F1, ROC-AUC, and Log Loss for both months.

### What we achieve after running it
- We can separate:
  - **input drift**
  - **score drift**
  - **performance drift**
- This gives a much more useful monitoring picture than watching only one metric.

### Remember
Monitoring happens in stages:

**1. Inputs arrive → check schema and data drift**

**2. Predictions are produced → monitor score distribution and flagged rate**

**3. Labels arrive later → monitor actual model performance**

AUC can remain acceptable while recall or calibration becomes poor, so monitor more than one metric.


In [ ]:
X_ref = reference.drop(columns=[LABEL])
X_drift = drifted.drop(columns=[LABEL])

p_ref = predict_model(
    best,
    data=X_ref,
    raw_score=True,
    verbose=False
)["prediction_score_1"].to_numpy()

p_drift = predict_model(
    best,
    data=X_drift,
    raw_score=True,
    verbose=False
)["prediction_score_1"].to_numpy()

flag_ref = (p_ref >= 0.5).astype(int)
flag_drift = (p_drift >= 0.5).astype(int)

print(
    "Mean predicted risk:",
    round(p_ref.mean(), 3),
    "->",
    round(p_drift.mean(), 3)
)

print(
    "Flagged share:",
    round(flag_ref.mean(), 3),
    "->",
    round(flag_drift.mean(), 3)
)

performance = pd.concat([
    report_card(
        reference[LABEL],
        flag_ref,
        p_ref,
        "reference month"
    ),
    report_card(
        drifted[LABEL],
        flag_drift,
        p_drift,
        "drifted month"
    )
])

display(performance.round(3))


## 9. Retrain Using the Same Recipe

### What we saw before
- We detected a change in the incoming data.
- We also measured how model predictions and performance changed.

### How this cell is different
- Instead of only monitoring the problem, we now show the retraining path.

### What this cell does
- Adds the newly labelled drifted month to the original training data.
- Creates a fresh `ClassificationExperiment`.
- Uses the same target, seed, folds, and model ID.
- `create_model(best_id)` recreates the same type of estimator using the updated data.

### What we achieve after running it
- We get a new version of the same model recipe trained on more recent data.
- The training process is reproducible instead of being manually changed during retraining.

### Remember
A good retraining process keeps the **recipe** under version control:

- feature definitions
- target definition
- `setup()` configuration
- model ID
- random seed
- package versions

The saved pickle is the output.

**The training recipe is the source.**


In [ ]:
retrain_data = pd.concat(
    [
        train_df,
        drifted
    ],
    ignore_index=True
)

retrain_exp = ClassificationExperiment()

retrain_exp.setup(
    data=retrain_data,
    target=LABEL,
    session_id=RANDOM_STATE,
    fold=2,
    verbose=False,
    html=False
)

retrained_model = retrain_exp.create_model(
    best_id,
    fold=2,
    verbose=False
)

retrain_cv = retrain_exp.pull()

print(
    "Retrained model:",
    best_name,
    f"(id='{best_id}')"
)

display(
    retrain_cv.loc[
        ["Mean", "Std"],
        ["Accuracy", "AUC", "Recall", "Prec.", "F1"]
    ]
)


# Final Revision Summary

## Complete PyCaret MLOps lifecycle

### 1. Train and select

```python
setup(...)
best = compare_models(...)
```

### 2. Freeze honest performance

```python
predict_model(best)
before = pull()
```

### 3. Finalize for deployment

```python
final = finalize_model(best)
```

### 4. Persist the pipeline

```python
save_model(final, "model_name")
loaded = load_model("model_name")
```

### 5. Score new data

```python
predict_model(
    loaded,
    data=new_rows,
    raw_score=True
)
```

### 6. Monitor

Check:

- input/schema quality
- PSI or other drift measures
- prediction-score distribution
- flagged rate
- model metrics after labels arrive

### 7. Retrain

```python
new_exp = ClassificationExperiment()
new_exp.setup(updated_data, ...)
new_model = new_exp.create_model(best_id)
```

---

## Additional PyCaret MLOps capabilities

PyCaret also provides tools for:

| Capability | PyCaret concept |
|---|---|
| Experiment tracking | `log_experiment=True`, MLflow and custom loggers |
| API scaffolding | `create_api()` |
| Docker scaffolding | `create_docker()` |
| Interactive app | `create_app()` |
| Fairness auditing | `check_fairness()` |
| Drift report | `check_drift()` |

These are useful extensions, but the essential lifecycle remains:

**evaluate honestly → finalize → persist → serve → monitor → retrain**

---

## Most important takeaways

- Never report holdout metrics after `finalize_model()`.
- Save the complete preprocessing + model pipeline.
- Verify that a loaded model reproduces the original predictions.
- Use `prediction_score_1` when the positive-class probability is required.
- Model monitoring starts before labels arrive.
- Input drift does not automatically mean model failure.
- Monitor score distributions and business threshold metrics as well as AUC.
- Retraining should reproduce the same model recipe on newer labelled data.
